In [2]:
# ===== STEP 1: Build the modelling dataset =====
import os, re, glob, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive
drive.mount("/content/drive")

PROJECT  = "/content/drive/MyDrive/IoT_Anomaly_Detection_Project"
PQ_DIR   = f"{PROJECT}/Data/CIC-IoT-DIAD-2024-parquet"
WORK     = f"{PROJECT}/Data/CIC-IoT-DIAD-2024-work"
EDA_DIR  = f"{PROJECT}/eda_csvs"
FLOW_DIR = f"{PQ_DIR}/Anomaly Detection - Flow Based features"
PARTS    = f"{WORK}/model_parts"            # one small file per flow file (lets you resume)
OUT      = f"{WORK}/model_data.parquet"     # the final modelling dataset

ATTACK_PER_FILE   = 2_000    # max attack flows kept per device from each attack file
ATTACK_PER_DEVICE = 20_000   # max attack flows per device in the end (split across attack types)
SEED    = 42
ID_COLS = ["Flow ID", "Src IP", "Dst IP", "Src Port", "Dst Port", "Timestamp", "Label"]
os.makedirs(PARTS, exist_ok=True)
rng = np.random.default_rng(SEED)

def find_file(name):
    """Look for a file in eda_csvs first, then in the work folder."""
    for folder in [EDA_DIR, WORK]:
        path = f"{folder}/{name}"
        if os.path.exists(path):
            print(f"Using {path}")
            return path
    raise FileNotFoundError(f"{name} not found in {EDA_DIR} or {WORK}")

# ---------- 1. Devices and their tiers ----------
tiers = pd.read_csv(find_file("device_tiers.csv"))
tiers = tiers[tiers["tier"] != "excluded"][["device", "device_type", "tier"]]
devices = set(tiers["device"])
print("Devices used:", len(devices))
print(tiers["tier"].value_counts().to_string(), "\n")

# ---------- 2. Address book (IP -> device), same rules as before ----------
everyone = pd.read_csv(find_file("device_ip_map_benign.csv"), index_col=0)
ip_count = {}
for ips in everyone["lan_ips"].dropna():
    for ip in ips.split(", "):
        ip_count[ip] = ip_count.get(ip, 0) + 1
book = everyone[(everyone["kind"] == "named device") & everyone["lan_ips"].notna()]
ip2dev = {ip: dev for dev, ips in book["lan_ips"].items()
          for ip in ips.split(", ") if ip_count[ip] == 1 and dev in devices}
print(f"IP addresses mapped: {len(ip2dev)} for {len(set(ip2dev.values()))} devices\n")

# ---------- 3. Go through every flow file ----------
files = sorted(glob.glob(f"{FLOW_DIR}/**/*.parquet", recursive=True))
print(f"Flow files found: {len(files)}  (expected 132)\n")
start = time.time()
for i, p in enumerate(files, 1):
    rel  = os.path.relpath(p, FLOW_DIR)
    part = f"{PARTS}/{i:03d}.parquet"
    if os.path.exists(part):
        continue                                            # already done in an earlier run
    family    = rel.split(os.sep)[0]
    is_benign = family.lower().startswith("benign")
    family    = "Benign" if is_benign else family
    attack    = "Benign" if is_benign else re.sub(r"[\d\-_]+$", "",
                 os.path.basename(p)[:-len(".parquet")].replace(".pcap_Flow", ""))

    pf = pq.ParquetFile(p)
    benign_parts, keep = [], {}
    if "Src IP" in pf.schema_arrow.names:
        for b in pf.iter_batches(batch_size=200_000):
            df = b.to_pandas()
            s = df["Src IP"].astype(str).str.strip().map(ip2dev)
            d = df["Dst IP"].astype(str).str.strip().map(ip2dev)
            d = d.where(d != s)                             # don't add the same device twice
            tagged = pd.concat([df[s.notna()].assign(device=s[s.notna()]),
                                df[d.notna()].assign(device=d[d.notna()])], ignore_index=True)
            if tagged.empty:
                continue
            tagged = tagged.drop(columns=[c for c in ID_COLS if c in tagged.columns])
            if is_benign:
                benign_parts.append(tagged)
            else:                                           # random sample across the whole file
                tagged["_r"] = rng.random(len(tagged))
                for dev, g in tagged.groupby("device"):
                    pool = pd.concat([keep[dev], g]) if dev in keep else g
                    keep[dev] = pool.nsmallest(ATTACK_PER_FILE, "_r")

    chunks = benign_parts if is_benign else list(keep.values())
    if chunks:
        out = pd.concat(chunks, ignore_index=True).drop(columns=["_r"], errors="ignore")
        feat = [c for c in out.columns if c != "device"]
        out[feat] = out[feat].apply(pd.to_numeric, errors="coerce").astype("float32")
        out[feat] = out[feat].replace([np.inf, -np.inf], np.nan)
        out["family"], out["attack_type"] = family, attack
        out["source_file"], out["is_attack"] = rel, int(not is_benign)
    else:
        out = pd.DataFrame({"device": pd.Series(dtype="string")})   # empty marker = file done
    out.to_parquet(part, index=False)
    print(f"[{i}/{len(files)}] {'BENIGN' if is_benign else 'attack'}  kept {len(out):>7,}  {rel}"
          f"   ({(time.time()-start)/60:.0f} min)")

# ---------- 4. Join everything, cap attacks per device ----------
parts = [pd.read_parquet(f) for f in sorted(glob.glob(f"{PARTS}/*.parquet"))]
data  = pd.concat([x for x in parts if len(x)], ignore_index=True)
ben, att = data[data["is_attack"] == 0], data[data["is_attack"] == 1]

pieces = []
for dev, g in att.groupby("device"):
    per_type = max(1, ATTACK_PER_DEVICE // g["attack_type"].nunique())   # equal share per attack type
    for _, gt in g.groupby("attack_type"):
        pieces.append(gt.sample(min(len(gt), per_type), random_state=SEED))
att = pd.concat(pieces, ignore_index=True)

# ---------- 5. Add tiers, drop useless columns, save ----------
data = pd.concat([ben, att], ignore_index=True).merge(tiers, on="device", how="inner")
meta = ["device", "device_type", "tier", "family", "attack_type", "source_file", "is_attack"]
feat = [c for c in data.columns if c not in meta]
const = [c for c in feat if data[c].nunique(dropna=False) <= 1]
data = data.drop(columns=const)
data.to_parquet(OUT, index=False)

# ---------- 6. Summary ----------
print(f"\n✅ Saved {len(data):,} rows × {len(feat) - len(const)} features → {OUT}")
print(f"Dropped {len(const)} constant columns: {const}")
print(f"Benign: {(data.is_attack == 0).sum():,}   Attack: {(data.is_attack == 1).sum():,}\n")
summary = data.groupby(["tier", "device_type", "device"]).agg(
    benign=("is_attack", lambda x: int((x == 0).sum())),
    attack=("is_attack", "sum"),
    attack_types=("attack_type", lambda x: x[x != "Benign"].nunique()),
).sort_values(["tier", "device_type", "benign"], ascending=[False, True, False])
pd.set_option("display.max_rows", 100, "display.width", 200)
print(summary.to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using /content/drive/MyDrive/IoT_Anomaly_Detection_Project/eda_csvs/device_tiers.csv
Devices used: 53
tier
per-device     43
device-type     9
global-only     1 

Using /content/drive/MyDrive/IoT_Anomaly_Detection_Project/eda_csvs/device_ip_map_benign.csv
IP addresses mapped: 57 for 53 devices

Flow files found: 132  (expected 132)

[1/132] BENIGN  kept  94,048  Benign/BenignTraffic.pcap_Flow.parquet   (0 min)
[2/132] BENIGN  kept  76,033  Benign/BenignTraffic1.pcap_Flow.parquet   (0 min)
[3/132] BENIGN  kept  84,195  Benign/BenignTraffic2.pcap_Flow.parquet   (0 min)
[4/132] BENIGN  kept  36,252  Benign/BenignTraffic3.pcap_Flow.parquet   (0 min)
[5/132] attack  kept   1,504  BruteForce/DictionaryBruteForce.pcap_Flow.parquet   (0 min)
[6/132] attack  kept   5,747  DDoS/DDoS ACK Fragmentation/DDoS-ACK_Fragmentation.pcap_Flow.parquet   (0 min)
[7/132] attack  ke

In [3]:
# =====  Train / validation / test split + scoring functions =====
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

PROJECT   = "/content/drive/MyDrive/IoT_Anomaly_Detection_Project"
WORK      = f"{PROJECT}/Data/CIC-IoT-DIAD-2024-work"
DATA      = f"{WORK}/model_data.parquet"
SPLIT_OUT = f"{WORK}/model_data_split.parquet"
SEED      = 42
META      = ["device", "device_type", "tier", "family", "attack_type", "source_file", "is_attack", "split"]

# ---------- 1. Load (or reuse the saved split, so it never changes) ----------
if os.path.exists(SPLIT_OUT):
    data = pd.read_parquet(SPLIT_OUT)
    print("Loaded existing split:", SPLIT_OUT)
else:
    data = pd.read_parquet(DATA)

    # ---------- 2. Split every device separately: 60% train / 20% val / 20% test ----------
    def strat(y):
        """Keep the benign/attack mix the same in each part (if there are enough of each)."""
        return y if np.bincount(y, minlength=2).min() >= 2 else None

    split = pd.Series("train", index=data.index)
    for dev, g in data.groupby("device"):
        idx, y = g.index.values, g["is_attack"].values.astype(int)
        rest, test = train_test_split(idx, test_size=0.20, stratify=strat(y), random_state=SEED)
        y_rest = data.loc[rest, "is_attack"].values.astype(int)
        train, val = train_test_split(rest, test_size=0.25, stratify=strat(y_rest), random_state=SEED)  # 0.25 of 80% = 20%
        split.loc[val], split.loc[test] = "val", "test"
    data["split"] = split.values
    data.to_parquet(SPLIT_OUT, index=False)
    print("✅ Split saved:", SPLIT_OUT)

FEATURES = [c for c in data.columns if c not in META]
print(f"{len(data):,} flows | {len(FEATURES)} features | {data['device'].nunique()} devices\n")

# ---------- 3. How many flows per device in each part ----------
counts = (data.groupby(["device", "split"])["is_attack"]
              .agg(benign=lambda x: int((x == 0).sum()), attack="sum")
              .unstack("split").fillna(0).astype(int))
counts.columns = [f"{kind}_{part}" for kind, part in counts.columns]
counts = counts.join(data.groupby("device")[["tier", "device_type"]].first())
counts["small_test"] = (counts["benign_test"] < 50) | (counts["attack_test"] < 30)
cols = ["tier", "device_type", "benign_train", "attack_train", "benign_val", "attack_val",
        "benign_test", "attack_test", "small_test"]
pd.set_option("display.max_rows", 100, "display.width", 220)
print(counts[cols].sort_values(["tier", "benign_test"], ascending=[False, False]).to_string())
small = counts.index[counts["small_test"]].tolist()
print(f"\n⚠️ Devices with a small test set ({len(small)}): {small}")

# ---------- 4. Leakage check: test flows that are exact copies of a training flow ----------
h = pd.util.hash_pandas_object(data[FEATURES].fillna(-999), index=False)
train_hashes = set(h[data["split"] == "train"])
is_test = data["split"] == "test"
dup = h[is_test].isin(train_hashes)
print(f"\nTest flows identical to a training flow: {dup.mean()*100:.1f}%  "
      f"(benign {dup[data.loc[is_test, 'is_attack'] == 0].mean()*100:.1f}%, "
      f"attack {dup[data.loc[is_test, 'is_attack'] == 1].mean()*100:.1f}%)")

# ---------- 5. Scoring functions used by EVERY model later ----------
PREVALENCES = [0.05, 0.01]   # "realistic" attack shares for PR-AUC
FPR_TARGET  = 0.01           # alarm threshold: about 1% of normal traffic flagged

def threshold_from_val(val_scores, val_y, fpr=FPR_TARGET):
    """Score above which an alarm is raised, set so ~1% of benign VALIDATION flows trigger it."""
    benign = np.asarray(val_scores)[np.asarray(val_y) == 0]
    return np.quantile(benign, 1 - fpr) if len(benign) else np.inf

def weighted_pr_auc(y, scores, prevalence):
    """PR-AUC as if attacks were `prevalence` of all traffic (attack flows get a smaller weight)."""
    y, scores = np.asarray(y), np.asarray(scores)
    n_b, n_a = (y == 0).sum(), (y == 1).sum()
    if n_a == 0 or n_b == 0:
        return np.nan
    w = np.where(y == 1, prevalence / (1 - prevalence) * n_b / n_a, 1.0)
    return average_precision_score(y, scores, sample_weight=w)

def evaluate(y, scores, threshold):
    """All metrics for one model on one device's test flows. Higher score = more suspicious."""
    y, scores = np.asarray(y), np.asarray(scores)
    out = {f"pr_auc_{int(p*100)}pct": weighted_pr_auc(y, scores, p) for p in PREVALENCES}
    alarm = scores > threshold
    out["recall"]   = alarm[y == 1].mean() if (y == 1).any() else np.nan
    out["fpr"]      = alarm[y == 0].mean() if (y == 0).any() else np.nan
    out["n_benign"] = int((y == 0).sum())
    out["n_attack"] = int((y == 1).sum())
    return out

# ---------- 6. Sanity test: random guessing should give PR-AUC ≈ 0.05 and ≈ 0.01 ----------
rng = np.random.default_rng(SEED)
test = data[is_test]
fake_scores = rng.random(len(test))
val = data[data["split"] == "val"]
thr = threshold_from_val(rng.random(len(val)), val["is_attack"])
print("\nRandom-guess sanity check:", {k: round(v, 3) if isinstance(v, float) else v
                                      for k, v in evaluate(test["is_attack"], fake_scores, thr).items()})
print("Expected: pr_auc_5pct ≈ 0.05, pr_auc_1pct ≈ 0.01, recall ≈ 0.01, fpr ≈ 0.01")

✅ Split saved: /content/drive/MyDrive/IoT_Anomaly_Detection_Project/Data/CIC-IoT-DIAD-2024-work/model_data_split.parquet
619,016 flows | 70 features | 53 devices

                                                   tier         device_type  benign_train  attack_train  benign_val  attack_val  benign_test  attack_test  small_test
device                                                                                                                                                               
Home Eye Camera                              per-device              Camera         30673          4637       10225        1546        10225         1546       False
harman kardon (Ampak Technology)             per-device       Smart speaker         17237          1942        5746         648         5746          648       False
Wyze Camera                                  per-device              Camera         16709          3416        5570        1139         5570         1139       False
LG Smar

In [ ]:
# ===== Train models at 3 levels and score every device =====
import os, time
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from xgboost import XGBClassifier

RESULTS = f"{WORK}/results_obj2.csv"
SEEDS   = [0, 1, 2 ]                      # start with 1 seed; later change to [0, 1, 2] and rerun (done parts are skipped)
MODELS  = ["IsolationForest", "RandomForest", "XGBoost"]

# ---------- 1. Every model to train: (level, scope name, which rows) ----------
scopes = [("global", "all devices", np.ones(len(data), dtype=bool))]
type_sizes = data.groupby("device_type")["device"].nunique()
for t in sorted(type_sizes[type_sizes >= 2].index):
    scopes.append(("device-type", t, (data["device_type"] == t).values))
for dev in sorted(data.loc[data["tier"] == "per-device", "device"].unique()):
    scopes.append(("per-device", dev, (data["device"] == dev).values))
print(f"Models per algorithm: {len(scopes)}  "
      f"(1 global + {sum(s[0]=='device-type' for s in scopes)} device-type + "
      f"{sum(s[0]=='per-device' for s in scopes)} per-device)\n")

# ---------- 2. How each algorithm is built (same settings at every level) ----------
def make_model(name, seed, n_benign, n_attack):
    if name == "IsolationForest":
        return IsolationForest(n_estimators=200, random_state=seed, n_jobs=-1)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=2, class_weight="balanced",
                                      random_state=seed, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                             colsample_bytree=0.8, tree_method="hist", eval_metric="aucpr",
                             scale_pos_weight=n_benign / max(n_attack, 1), random_state=seed, n_jobs=-1)

def train_scorer(name, seed, train):
    """Train one model; return a function that gives a 'suspicion score' (higher = more likely attack)."""
    med = train[FEATURES].median()
    prep = lambda df: df[FEATURES].fillna(med).fillna(0)
    y = train["is_attack"].values
    if name == "IsolationForest":                          # learns normal traffic only
        m = make_model(name, seed, 0, 0).fit(prep(train)[y == 0])
        return lambda df: -m.score_samples(prep(df))
    m = make_model(name, seed, (y == 0).sum(), (y == 1).sum()).fit(prep(train), y)
    return lambda df: m.predict_proba(prep(df))[:, 1]

# ---------- 3. Resume: what is already done? ----------
done = set()
if os.path.exists(RESULTS):
    prev = pd.read_csv(RESULTS)
    done = set(zip(prev["seed"], prev["model"], prev["level"], prev["scope"]))
    print(f"Already done: {len(done)} trained models (skipped)\n")

info = data.groupby("device")[["tier", "device_type"]].first()
start = time.time()

# ---------- 4. Train, set threshold on validation, score each device's test flows ----------
for seed in SEEDS:
    for name in MODELS:
        for level, scope, mask in scopes:
            if (seed, name, level, scope) in done:
                continue
            sub = data[mask]
            train = sub[sub["split"] == "train"]
            val   = sub[sub["split"] == "val"]
            test  = sub[sub["split"] == "test"]
            score = train_scorer(name, seed, train)
            thr   = threshold_from_val(score(val), val["is_attack"])
            test  = test.assign(score=score(test))

            rows = []
            for dev, g in test.groupby("device"):
                rows.append({"seed": seed, "model": name, "level": level, "scope": scope, "device": dev,
                             "tier": info.loc[dev, "tier"], "device_type": info.loc[dev, "device_type"],
                             **evaluate(g["is_attack"], g["score"], thr)})
            pd.DataFrame(rows).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
            print(f"seed {seed} | {name:15s} | {level:11s} | {scope[:35]:35s} | "
                  f"{len(train):>7,} train rows | {(time.time()-start)/60:5.1f} min")

# ---------- 5. First comparison: the 43 devices that have all 3 levels ----------
res = pd.read_csv(RESULTS)
res = res[res["tier"] == "per-device"]
summary = (res.groupby(["model", "level"])[["pr_auc_5pct", "pr_auc_1pct", "recall", "fpr"]]
              .mean().round(3)
              .reindex(pd.MultiIndex.from_product([MODELS, ["global", "device-type", "per-device"]])))
print("\nAverage over the 43 per-device-tier devices (test set):")
print(summary.to_string())


Models per algorithm: 51  (1 global + 7 device-type + 43 per-device)

Already done: 153 trained models (skipped)

seed 1 | IsolationForest | global      | all devices                         | 371,371 train rows |   0.1 min
seed 1 | IsolationForest | device-type | Camera                              | 125,835 train rows |   0.1 min
seed 1 | IsolationForest | device-type | Hub / bridge                        |  41,848 train rows |   0.1 min
seed 1 | IsolationForest | device-type | Lighting                            |  27,239 train rows |   0.1 min
seed 1 | IsolationForest | device-type | Plug / switch                       |  55,166 train rows |   0.1 min
seed 1 | IsolationForest | device-type | Sensor / appliance                  |  25,897 train rows |   0.2 min
seed 1 | IsolationForest | device-type | Smart speaker                       |  73,631 train rows |   0.2 min
seed 1 | IsolationForest | device-type | TV / streaming                      |  14,763 train rows |   0.2 min
seed 1

apparently one-size-fits-all fails badly for unsupervised detection. kjo ishte te seed 1